# F0 recovery · Frozen DINOv2 + MLP-BN-Dropout
Recreate the augmentation-seed-123, classifier-seed-2026 checkpoint required by H2.
This is a new training run, not recovery of the lost original weights. The reported
98% result is not guaranteed. No H0/H1 experiments are run.

In Colab select a T4 GPU, configure the paths below, and Run all. By default outputs
persist in Google Drive. Enable Colab Secret `KAGGLE_API_TOKEN` and accept the competition
rules on Kaggle. Alternatively disable downloading and provide extracted data.
The checkpoint filename and metadata are compatible with H2. Existing artifacts are
protected by a new run directory. No training outputs are stored in this notebook.

## 0. Setup

In [ ]:
%pip -q install timm scikit-learn pandas pillow tqdm matplotlib kagglehub

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import uuid
import random, hashlib, json
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from torchvision.transforms import InterpolationMode
from PIL import Image
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
from IPython.display import display

USE_DRIVE = True
DOWNLOAD_FROM_KAGGLE = True
DATA_ROOT = Path('/content/car_data')
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/Car-Classification-Benchmark') if USE_DRIVE else Path('/content')
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid.uuid4().hex[:8]
OUTPUT_DIR = ROOT / 'student_outputs_F0_recovery' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
if DOWNLOAD_FROM_KAGGLE:
    import os, kagglehub
    if not os.environ.get('KAGGLE_API_TOKEN'):
        from google.colab import userdata
        os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')
    DATA_ROOT = Path(kagglehub.competition_download('idnn-swe-3052-assignment-3-flowers-classification'))

SEED_AUG = 123
SEED_HEAD = 2026
random.seed(SEED_AUG)
np.random.seed(SEED_AUG)
torch.manual_seed(SEED_AUG)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED_AUG)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32 if DEVICE == 'cuda' else 16
EPOCHS = 35
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-3
PATIENCE = 7
for name in ('train', 'val', 'test'):
    assert (DATA_ROOT / name).is_dir(), f'Missing {DATA_ROOT / name}'
print('Data:', DATA_ROOT, '| Outputs:', OUTPUT_DIR, '| Device:', DEVICE)

## 1. Original F0 transforms and class mapping

In [ ]:
# preprocessing change
MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0),
        ratio=(0.9, 1.1),
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])

val_transform = transforms.Compose([
    transforms.Resize(
        256,
        interpolation=InterpolationMode.BICUBIC
    ),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)
])


train_ds = datasets.ImageFolder(DATA_ROOT / 'train', transform=train_transform)
val_ds = datasets.ImageFolder(DATA_ROOT / 'val', transform=val_transform)
preprocess = val_transform

assert train_ds.class_to_idx == val_ds.class_to_idx, 'train/ and val/ class folders must match'
NUM_CLASSES = len(train_ds.classes)
print('Classes:', NUM_CLASSES, '| Train:', len(train_ds), '| Val:', len(val_ds))
print('First class folders:', train_ds.classes[:5])
assert NUM_CLASSES == 20, f'Expected 20 classes, got {NUM_CLASSES}'

# Class-folder names from the instructor's export are prefixed by 000_, 001_, ...
# These prefixes make ImageFolder indices equal the Kaggle integer labels.
for i, folder in enumerate(train_ds.classes):
    assert folder.startswith(f'{i:03d}_'), f'Class-label mismatch at {folder}; ask your instructor'

assert (len(train_ds), len(val_ds)) == (700, 200), 'Expected original 700/200 split.'

## 2. Test IDs (no predictions or Kaggle submission generated)

In [ ]:
EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
test_dir = DATA_ROOT / 'test'
test_files = sorted(p for p in test_dir.iterdir() if p.is_file() and p.suffix.lower() in EXTENSIONS)
assert test_files, f'No images found inside {test_dir.resolve()}'

sample_path = DATA_ROOT / 'sample_submission.csv'
assert sample_path.exists(), 'Original sample_submission.csv is required.'
if sample_path.exists():
    sample = pd.read_csv(sample_path, dtype={'image_id': str})
    assert {'image_id','label'}.issubset(sample.columns)
    required_ids = sample['image_id'].astype(str).tolist()
else:
    required_ids = sorted(p.stem for p in test_files)

by_id = {p.stem: p for p in test_files}
assert len(by_id) == len(test_files), 'Two test filenames share the same image_id stem'
assert len(set(required_ids)) == len(required_ids), 'Duplicate image_id in sample_submission.csv'
assert set(required_ids) == set(by_id), ('Test image IDs do not match sample_submission.csv',
                                         len(set(required_ids)-set(by_id)), len(set(by_id)-set(required_ids)))
print('Unlabeled test images:', len(required_ids))
print('Submission ID example:', required_ids[:3])

class TestImages(Dataset):
    def __init__(self, ids): self.ids = ids
    def __len__(self): return len(self.ids)
    def __getitem__(self, index):
        p = by_id[self.ids[index]]
        with Image.open(p) as im: image = preprocess(im.convert('RGB'))
        return image

test_ds = TestImages(required_ids)

## 3. Original pretrained, frozen DINOv2

In [ ]:

# Experiment F0 — DINOv2 ViT-S/14

MODEL_NAME = "dinov2_vits14"

backbone = torch.hub.load(
    "facebookresearch/dinov2",
    MODEL_NAME
).to(DEVICE)

# Frozen feature extractor
backbone.eval()

for param in backbone.parameters():
    param.requires_grad_(False)

FEATURE_DIM = backbone.embed_dim

assert FEATURE_DIM == 384

# Verify output shape
with torch.inference_mode():
    dummy = torch.randn(2, 3, 224, 224, device=DEVICE)
    features = backbone(dummy)

assert features.shape == (2, 384), features.shape

print("Backbone:", MODEL_NAME)
print("Feature dimension:", FEATURE_DIM)
print("Test output shape:", features.shape)

## 4. Extract original plus four augmented training views
This preserves F0's original offline multiview procedure and training-only statistics.
Validation/test images never enter classifier training or normalization fitting.
The per-run cache prevents accidental reuse of incompatible previous artifacts.

In [ ]:

# Experiment configuration
EXPERIMENT = "P3_geometry_multiview"
N_AUG = 4
INCLUDE_ORIGINAL = True
SEED = 123

# Reproducibility
random.seed(SEED_AUG)
np.random.seed(SEED_AUG)
torch.manual_seed(SEED_AUG)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_AUG)

# Cache signature
def dataset_signature():
    paths = (
        [Path(p) for p, _ in train_ds.samples + val_ds.samples]
        + [by_id[i] for i in required_ids]
    )

    parts = [
        MODEL_NAME,
        EXPERIMENT,
        repr(train_transform),
        repr(val_transform),
        str(N_AUG),
        str(INCLUDE_ORIGINAL),
        str(SEED),
        str(train_ds.class_to_idx)
    ]

    for p in paths:
        st = p.stat()
        parts.append(
            f'{p.resolve()}|{st.st_size}|{st.st_mtime_ns}'
        )

    return hashlib.sha256(
        '\n'.join(parts).encode()
    ).hexdigest()


CACHE = OUTPUT_DIR / (
    f"{MODEL_NAME}_{EXPERIMENT}_"
    f"augseed{SEED_AUG}_features.pt"
)
signature = dataset_signature()


@torch.inference_mode()
def extract(loader, desc):
    embeddings = []
    backbone.eval()

    for batch in tqdm(loader, desc=desc):
        x = batch[0] if isinstance(batch, (tuple, list)) else batch
        z = backbone(x.to(DEVICE)).float()

        assert z.ndim == 2 and z.shape[1] == FEATURE_DIM
        embeddings.append(z.cpu())

    return torch.cat(embeddings)


kwargs = dict(
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE == 'cuda')
)

# Check cache
if CACHE.exists():
    try:
        cached = torch.load(
            CACHE, map_location='cpu', weights_only=True
        )
    except TypeError:
        cached = torch.load(CACHE, map_location='cpu')
else:
    cached = None


if cached is not None and cached.get('signature') == signature:
    print("Reusing feature cache:", CACHE)

    X_train = cached['X_train']
    X_val = cached['X_val']
    X_test = cached['X_test']

else:
    train_views = []
    original_transform = train_ds.transform

    try:
        if INCLUDE_ORIGINAL:
            train_ds.transform = val_transform
            train_views.append(
                extract(
                    DataLoader(train_ds, **kwargs),
                    'Original training features'
                )
            )

        train_ds.transform = train_transform

        for i in range(N_AUG):
            train_views.append(
                extract(
                    DataLoader(train_ds, **kwargs),
                    f'Augmented view {i+1}/{N_AUG}'
                )
            )
    finally:
        train_ds.transform = original_transform

    X_train = torch.cat(train_views, dim=0)

    X_val = extract(
        DataLoader(val_ds, **kwargs),
        'Validation features'
    )

    X_test = extract(
        DataLoader(test_ds, **kwargs),
        'Test features'
    )

    torch.save({
        'signature': signature,
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test
    }, CACHE)


# Labels
NUM_VIEWS = N_AUG + int(INCLUDE_ORIGINAL)

y_train = torch.tensor(
    train_ds.targets, dtype=torch.long
).repeat(NUM_VIEWS)

y_val = torch.tensor(
    val_ds.targets, dtype=torch.long
)

assert len(X_train) == len(y_train)
assert len(X_val) == len(y_val)
assert len(X_test) == len(required_ids)

FEATURES_SIGNATURE = signature
FEATURES_MODEL_NAME = MODEL_NAME

print("Model:", MODEL_NAME)
print("Preprocessing:", EXPERIMENT)
print("Feature cache:", CACHE)
print("X_train:", X_train.shape)
print('y_train:', y_train.shape)
print('X_val:', X_val.shape)
print('X_test:', X_test.shape)

## 5. Training-only feature normalization

In [ ]:
feature_mean = X_train.mean(dim=0)
feature_std = X_train.std(dim=0, unbiased=False).clamp_min(1e-6)
Z_train = (X_train - feature_mean) / feature_std
Z_val = (X_val - feature_mean) / feature_std
Z_test = (X_test - feature_mean) / feature_std
print('Features standardized without validation/test leakage.')

## 6. Restore the original head architecture and train seed 2026

In [ ]:
HEAD_CONFIGS = {
    "E0_linear": {
        "hidden_dim": None,
        "use_bn": False,
        "dropout": 0.0
    },
    "E1_mlp": {
        "hidden_dim": 128,
        "use_bn": False,
        "dropout": 0.0
    },
    "E2_mlp_bn": {
        "hidden_dim": 128,
        "use_bn": True,
        "dropout": 0.0
    },
    "E3_mlp_bn_drop01": {
        "hidden_dim": 128,
        "use_bn": True,
        "dropout": 0.1
    },
    "E4_mlp_bn_drop03": {
        "hidden_dim": 128,
        "use_bn": True,
        "dropout": 0.3
    }
}


def build_head(experiment, input_dim, num_classes):
    cfg = HEAD_CONFIGS[experiment]

    if cfg["hidden_dim"] is None:
        return nn.Linear(input_dim, num_classes)

    layers = [
        nn.Linear(input_dim, cfg["hidden_dim"])
    ]

    if cfg["use_bn"]:
        layers.append(
            nn.BatchNorm1d(cfg["hidden_dim"])
        )

    layers.append(nn.ReLU())

    if cfg["dropout"] > 0:
        layers.append(nn.Dropout(cfg["dropout"]))

    layers.append(
        nn.Linear(cfg["hidden_dim"], num_classes)
    )

    return nn.Sequential(*layers)

In [ ]:
from torch.utils.data import TensorDataset
training_loader = DataLoader(TensorDataset(Z_train, y_train), batch_size=64,
                             shuffle=True, num_workers=0)
HEAD_EXPERIMENT = "E4_mlp_bn_drop03"

# Ensure features match the selected backbone
assert FEATURES_MODEL_NAME == MODEL_NAME
assert FEATURES_SIGNATURE == dataset_signature(), (
    "Feature configuration changed. "
    "Rerun Sections 5 and 6."
)

# Reset classifier randomness
random.seed(SEED_HEAD)
np.random.seed(SEED_HEAD)
torch.manual_seed(SEED_HEAD)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_HEAD)

# Reproducible training data order
loader_generator = torch.Generator()
loader_generator.manual_seed(SEED_HEAD)

training_loader = DataLoader(
    TensorDataset(Z_train, y_train),
    batch_size=64,
    shuffle=True,
    num_workers=0,
    generator=loader_generator
)

# Fresh classifier for each experiment
head = build_head(
    HEAD_EXPERIMENT,
    FEATURE_DIM,
    NUM_CLASSES
).to(DEVICE)

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    head.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)
print("Experiment:", HEAD_EXPERIMENT)
print("Head seed:", SEED_HEAD)
print("Architecture:")
print(head)

best_accuracy = -1.0
best_weights = None
bad_epochs = 0
history = []
for epoch in range(1, EPOCHS + 1):
    head.train()
    running_loss = 0.0
    for xb, yb in training_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(head(xb), yb)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * len(yb)
    head.eval()
    with torch.inference_mode():
        val_pred = head(Z_val.to(DEVICE)).argmax(dim=1).cpu().numpy()
    val_accuracy = accuracy_score(y_val.numpy(), val_pred)
    history.append({'epoch': epoch, 'train_loss': running_loss / len(Z_train),
                    'val_accuracy': val_accuracy})
    print(f'Epoch {epoch:02d}/{EPOCHS} | train loss: {history[-1]["train_loss"]:.4f} | val accuracy: {val_accuracy:.2%}')
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_weights = {key: value.detach().cpu().clone() for key, value in head.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print('Early stopping: validation accuracy did not improve')
            break

head.load_state_dict(best_weights)
head.eval()
print(f'Best validation accuracy: {best_accuracy:.2%}')
# pd.DataFrame(history).to_csv(OUTPUT_DIR / 'training_history.csv', index=False)

# torch.save({'head_state_dict': best_weights, 'feature_mean': feature_mean,
#             'feature_std': feature_std, 'class_to_idx': train_ds.class_to_idx,
#             'backbone': MODEL_NAME}, OUTPUT_DIR / 'linear_head.pt')

RUN_NAME = (
    f"dinov2_p3_e4_aug{SEED_AUG}_head{SEED_HEAD}"
)

checkpoint_path = OUTPUT_DIR / f"{RUN_NAME}.pt"
submission_path = OUTPUT_DIR / f"{RUN_NAME}.csv"

pd.DataFrame(history).to_csv(
    OUTPUT_DIR / f"{RUN_NAME}_history.csv",
    index=False
)

torch.save({
    "head_state_dict": best_weights,
    "feature_mean": feature_mean,
    "feature_std": feature_std,
    "class_to_idx": train_ds.class_to_idx,
    "backbone": MODEL_NAME,
    "preprocessing": EXPERIMENT,
    "head_experiment": HEAD_EXPERIMENT,
    "head_config": HEAD_CONFIGS[HEAD_EXPERIMENT],
    "seed_aug": SEED_AUG,
    "seed_head": SEED_HEAD,
    "best_accuracy": best_accuracy
}, OUTPUT_DIR / f"{RUN_NAME}.pt")

print("Saved experiment:", RUN_NAME)


## 7. Evaluate the newly trained F0 classifier

In [ ]:
head.eval()
with torch.inference_mode():
    val_prediction = head(Z_val.to(DEVICE)).argmax(1).cpu().numpy()
accuracy = float(accuracy_score(y_val.numpy(), val_prediction))
macro_f1 = float(f1_score(y_val.numpy(), val_prediction, average='macro', zero_division=0))
print(f'Recreated F0 validation accuracy: {accuracy:.2%} | macro-F1: {macro_f1:.4f}')
reproduced = bool(np.isclose(accuracy, 0.98, rtol=0, atol=1e-8)
                  and np.isclose(macro_f1, 0.9803, rtol=0, atol=5e-4))
checkpoint = OUTPUT_DIR / 'dinov2_p3_e4_aug123_head2026.pt'
saved = torch.load(checkpoint, map_location='cpu', weights_only=True)
assert saved['seed_head'] == 2026 and saved['seed_aug'] == 123
assert saved['class_to_idx'] == train_ds.class_to_idx
assert saved['feature_mean'].numel() == saved['feature_std'].numel() == 384
assert saved['head_config'] == HEAD_CONFIGS['E4_mlp_bn_drop03']
check_head = build_head('E4_mlp_bn_drop03', 384, 20)
check_head.load_state_dict(saved['head_state_dict'], strict=True)
summary = {'accuracy': accuracy, 'macro_f1': macro_f1, 'matches_H2_baseline_gate': reproduced,
           'checkpoint': str(checkpoint), 'seed_aug': SEED_AUG, 'seed_head': SEED_HEAD,
           'torch': str(torch.__version__), 'source': 'F0 notebook commit 5a393fa56ae33a183a2268253528a795b5a450fe'}
(OUTPUT_DIR / 'F0_recovery_summary.json').write_text(json.dumps(summary, indent=2))
print('Saved checkpoint:', checkpoint)
print('Set H2 F0_CHECKPOINT to:')
print(f'F0_CHECKPOINT = Path({str(checkpoint)!r})')
if reproduced:
    print('Measured metrics match the expected H2 gate. H2 will independently verify them.')
else:
    print('Expected F0 metrics were NOT reproduced. Do not bypass the H2 gate. '
          'Send the recovery summary/history for diagnosis before fine-tuning.')
print('Keep this checkpoint in Drive or download it before ending the runtime.')
hist = pd.DataFrame(history)
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(hist.epoch, hist.val_accuracy * 100, marker='o')
ax.set(xlabel='Epoch', ylabel='Validation accuracy (%)', title='Recreated F0 classifier')
ax.grid(alpha=.25)
fig.savefig(OUTPUT_DIR / 'F0_validation_curve.png', dpi=150)
plt.show()